In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
df=pd.read_csv(r"C:\Users\Suyash\Downloads\online_retail.csv.zip")
print(df)

       InvoiceNo StockCode                          Description  Quantity  \
0         536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1         536365     71053                  WHITE METAL LANTERN         6   
2         536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3         536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4         536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   
...          ...       ...                                  ...       ...   
541904    581587     22613          PACK OF 20 SPACEBOY NAPKINS        12   
541905    581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
541906    581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
541907    581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
541908    581587     22138        BAKING SET 9 PIECE RETROSPOT          3   

                InvoiceDate  UnitPrice  CustomerID         Country  
0     

In [3]:
df=df.dropna(subset=['CustomerID'])
df=df[df['Quantity']>0]
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df['InvoiceDate'])

0        2010-12-01 08:26:00
1        2010-12-01 08:26:00
2        2010-12-01 08:26:00
3        2010-12-01 08:26:00
4        2010-12-01 08:26:00
                 ...        
541904   2011-12-09 12:50:00
541905   2011-12-09 12:50:00
541906   2011-12-09 12:50:00
541907   2011-12-09 12:50:00
541908   2011-12-09 12:50:00
Name: InvoiceDate, Length: 397924, dtype: datetime64[us]


In [4]:
import datetime as dt

reference_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,  # Recency
    'InvoiceNo': 'count',                                     # Frequency
    'TotalPrice': 'sum'                                       # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
scaler=StandardScaler()

X_scaler=scaler.fit_transform(rfm)

kMeans=KMeans(n_clusters=3,random_state=42)
rfm['Cluster']=kMeans.fit_predict(X_scaler)
print(rfm.groupby('Cluster').mean())

            Recency    Frequency       Monetary
Cluster                                        
0         41.368762   103.066235    2028.208836
1        247.308333    27.789815     637.318510
2          4.692308  2566.000000  126118.310000


In [6]:
df=df.merge(rfm['Cluster'],on='CustomerID')

top_products=df.groupby(['Cluster','Description'])['Quantity']\
               .sum().reset_index()
top_products = top_products.sort_values(['Cluster', 'Quantity'], ascending=[True, False])
print(top_products)

      Cluster                        Description  Quantity
3675        0  WORLD WAR 2 GLIDERS ASSTD DESIGNS     47543
1703        0            JUMBO BAG RED RETROSPOT     39623
210         0      ASSORTED COLOUR BIRD ORNAMENT     31005
2513        0                     POPCORN HOLDER     28419
2194        0    PACK OF 72 RETROSPOT CAKE CASES     24955
...       ...                                ...       ...
9505        2    WOOD AND GLASS MEDICINE CABINET         1
9534        2     WOODLAND LARGE BLUE FELT HEART         1
9536        2      WOODLAND LARGE RED FELT HEART         1
9540        2     WOODLAND SMALL BLUE FELT HEART         1
9605        2       ZINC HEARTS PLANT POT HOLDER         1

[9614 rows x 3 columns]


In [9]:
def recommend(customer_id):
    cluster=rfm.loc[customer_id,'Cluster']
    products=top_products[top_products['Cluster']==cluster] \
        .head(5)['Description'].tolist()

    return products

In [10]:
for i in range(4):
    print(f"\nTop products for Cluster {i}:")
    print(top_products[top_products['Cluster'] == i]
          .head(5)['Description'].values)


Top products for Cluster 0:
<StringArray>
['WORLD WAR 2 GLIDERS ASSTD DESIGNS',           'JUMBO BAG RED RETROSPOT',
     'ASSORTED COLOUR BIRD ORNAMENT',                    'POPCORN HOLDER',
   'PACK OF 72 RETROSPOT CAKE CASES']
Length: 5, dtype: str

Top products for Cluster 1:
<StringArray>
[    'MEDIUM CERAMIC TOP STORAGE JAR', 'FAIRY CAKE FLANNEL ASSORTED COLOUR',
 'WHITE HANGING HEART T-LIGHT HOLDER',  'WORLD WAR 2 GLIDERS ASSTD DESIGNS',
               'SMALL POPCORN HOLDER']
Length: 5, dtype: str

Top products for Cluster 2:
<StringArray>
[       'PAPER CRAFT , LITTLE BIRDIE',    'PACK OF 72 RETROSPOT CAKE CASES',
                 'RABBIT NIGHT LIGHT', 'WHITE HANGING HEART T-LIGHT HOLDER',
                'SPACEBOY LUNCH BOX ']
Length: 5, dtype: str

Top products for Cluster 3:
<StringArray>
[]
Length: 0, dtype: str


In [11]:
customer_id=rfm.index[0]
print("Customer ID",customer_id)
print("Cluster",rfm.loc[customer_id,'Cluster'])
print("Recommended Products:",recommend(customer_id))

Customer ID 12346.0
Cluster 1
Recommended Products: ['MEDIUM CERAMIC TOP STORAGE JAR', 'FAIRY CAKE FLANNEL ASSORTED COLOUR', 'WHITE HANGING HEART T-LIGHT HOLDER', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 'SMALL POPCORN HOLDER']
